# Chapter 1 — Does reinforcement learning rediscover Kelly?

Blackjack has a rare property for a betting problem: **we already know the optimal bet-sizing rule.** Once
you can count, the edge at each true count is measurable, and the growth-optimal wager is the **Kelly**
bet — a fixed fraction of bankroll, scaled by the edge. That makes this a clean test of a question that
recurs everywhere in applied ML: *can an end-to-end learner rediscover a rule we could also just derive?*

So we put three bettors on the same table, on identical terms (same basic-strategy play, same shoe, same
sessions), and measure them:

- **Flat** — bet one unit, always. The floor: no betting skill at all.
- **Kelly** — the analytic bet ladder, sized from the measured edge-by-count curve. The target.
- **BetAgent** — a DQN that sees `(true count, decks remaining, bankroll)` and learns a bet from reward.
  *"Does RL rediscover Kelly?"*

**The bottom line, up front:** it does not. Across seeds and both bankroll regimes, the learned bettor
**never reaches Kelly** — it converges to ≈ Flat in the growth regime, slightly *below* Flat in ruin — and
the near-Kelly bet curves it *transiently* produces are, when measured, *worse* than Flat. The one policy
that beats Flat is the **analytic Kelly**. The rest
of this report is *why* — a thin, noisy edge that end-to-end value learning cannot extract, shown by
ruling the alternatives out (including a hypothesis of our own that we built a test for and falsified).

### The chapters

1. **Does RL rediscover Kelly?** — the scoreboard and how to read it. *(this chapter)*
2. **The learned bettor, up close** — training dynamics, the multi-seed four-axis result (it ≈ Flat).
3. **Why not — the thin edge** — the prize quantified, the orbit that *visits* Kelly but won't hold it,
   and why the visited ramps are *worse* than Flat.
4. **A hypothesis, tested and falsified** — "it keyed on wealth, not edge" (the embedding) → two controlled
   tests (drop wealth from the input; train across the wealth axis) → the wall is fundamental, not
   representational.
5. **Synthesis** — structured ≫ end-to-end on a sub-noise signal; the honesty trail; what we'd try next.

In [1]:
import sys; sys.path.insert(0, '.')
import pandas as pd

from blackjack_rl.analysis_loader import load_bet_evals, ladder_baselines, ladder_provenance, show

ev = load_bet_evals()
fin = ev[(ev.phase == "final") & (ev.regime == ev.train_regime)]   # each agent scored in its native regime
base = ladder_baselines()   # the TIGHT 20k Kelly/Flat baseline — the agent's own 2k eval is far too noisy
                            # for these sub-0.2e-4 gaps (its cached Kelly even lands on the wrong side of 0).


# headline agent config per regime: growth = raw encoder; ruin = the clean cell (gamma 0.95, double off)
AGENT_FILT = {"growth": dict(bankroll_feature="raw"), "ruin": dict(gamma=0.95, double="off")}


def agent_cell(regime, **filt):
    m = fin[(fin.train_regime == regime) & (fin.bettor == "agent")]
    for k, v in filt.items():
        m = m[m[k] == v]
    return m


# Error margins, the *honest* one per policy type:
#   agent      — SD across the 6 training seeds (does it hold seed-to-seed?)
#   Kelly/Flat — deterministic policies, no seed spread; the 20k ladder's own 95% CI half-width instead.
def pm(mu, margin):
    return f"{mu:+.2f} ± {margin:.2f}"


rows = []
for regime, filt in AGENT_FILT.items():
    a = agent_cell(regime, **filt)
    k = base[(base.regime == regime) & (base.bettor == "kelly")].iloc[0]
    f = base[(base.regime == regime) & (base.bettor == "flat")].iloc[0]
    rows += [
        {"regime": regime, "policy": "learned BetAgent", "seeds": a.seed.nunique(),
         "growth /hand (x1e-4)": pm(a.growth_1e4.mean(), a.growth_1e4.std()),
         "deep-drawdown %": f"{a.dd_pct.mean():.2f} ± {a.dd_pct.std():.2f}"},
        {"regime": regime, "policy": "analytic Kelly", "seeds": "—",
         "growth /hand (x1e-4)": pm(k.growth_1e4, (k.growth_hi_1e4 - k.growth_lo_1e4) / 2),
         "deep-drawdown %": f"{k.dd_pct:.2f}"},
        {"regime": regime, "policy": "Flat (floor)", "seeds": "—",
         "growth /hand (x1e-4)": pm(f.growth_1e4, (f.growth_hi_1e4 - f.growth_lo_1e4) / 2),
         "deep-drawdown %": f"{f.dd_pct:.2f}"},
    ]
show(pd.DataFrame(rows),
     caption="Scoreboard — growth rate and deep-drawdown by policy and regime (native cell)",
     source=ladder_provenance(role="Kelly/Flat = tight 20k ladder; agent ± = SD over 6 seeds, baselines ± = eval 95% CI"))

regime,policy,seeds,growth /hand (x1e-4),deep-drawdown %
growth,learned BetAgent,6,-0.19 ± 0.08,0.18 ± 0.29
growth,analytic Kelly,—,-0.05 ± 0.02,0.14
growth,Flat (floor),—,-0.15 ± 0.01,0.00
ruin,learned BetAgent,6,-0.48 ± 0.04,1.39 ± 0.44
ruin,analytic Kelly,—,-0.35 ± 0.03,2.33
ruin,Flat (floor),—,-0.39 ± 0.03,0.80


## 1.1 The headline

Read the scoreboard one column at a time.

- **Growth rate** (log-growth per hand, the money metric) — the striking fact is that *even the optimal
  bettor barely wins*. Analytic Kelly's growth is **−0.05 × 10⁻⁴/hand** — still **net-negative** (the
  table-minimum forces mandatory −EV bets), but it reliably clears Flat's −0.15 by ~**0.10 × 10⁻⁴**
  (growth regime; independent-sample z ≈ 7, decisive). In the ruin regime the Kelly-over-Flat edge is
  real but **marginal** (~0.05 × 10⁻⁴, p ≈ 0.04), and bought with more drawdown. The learned BetAgent
  captures none of it — landing **≈ Flat** in the growth regime (−0.19, n.s. vs Flat) and **significantly
  below Flat** in ruin (−0.48 vs −0.39, z ≈ 4 across seeds, with higher drawdown): no skill gained, and in
  ruin a small price paid for trying.
- **The prize is tiny.** The *entire* edge Kelly buys over Flat is about **0.10 × 10⁻⁴ per hand** — the
  whole signal an end-to-end learner would have to extract, and it captures **none** of it. Chapter 3
  makes the case that this number is *why*: the edge is smaller than the per-hand noise the agent
  estimates it through.

So the answer to the title is **no** — but "no" with a precise, honest reason, which is the point. (Note
that here "beats Flat" means *loses less*: in the growth regime every bettor is net-negative — the skill is
restraint, not profit.)

## 1.2 How to read the rest — the ladder and the four axes

Two habits carry the whole report.

**The ladder.** Every result compares the same three rungs — **Flat → Kelly → BetAgent** — on identical
terms. Flat is the floor (is *any* betting skill learned?); Kelly is the ceiling (the derivable optimum);
BetAgent is the question. "≈ Flat" means *no skill learned*; "≈ Kelly" would mean *rediscovered it*.

**The four axes.** A bettor is never one number. We hold four apart, because a policy can win one and lose
another:

- **growth rate ± CI** — long-run log-growth (the objective),
- **ruin %** — probability of going broke (the catastrophe),
- **deep-drawdown %** — probability of losing ≥ half the bankroll (the pain even short of ruin),
- **final-bankroll distribution** — where you actually end up.

The recurring trap this report keeps catching: a bet *curve* that *looks* like Kelly at one probe can have
terrible drawdown once measured. So we trust the **four-axis eval at the native bankroll**, never the
eyeballed curve. Keep the ladder and the four axes in mind and the rest reads cleanly.

## 1.3 How we measure

One place for the evaluation design, so every later number reads unambiguously:

- **Agent** — each config is trained under **6 random seeds**; every figure shows the seeds as **dots**
  (at n = 6 a symmetric SD whisker is ≈ the 95% CI and would dwarf the bar, so we plot the raw points).
  Each seed is scored over **2,000 sessions** on the four-axis suite.
- **Kelly / Flat** — deterministic policies (no seed spread), measured once at **20,000 sessions** (the
  committed bet-ladder) and shown with their **Monte-Carlo 95% CI**. This is the *tight* baseline the
  scoreboard compares against — the agent's own 2k eval is too noisy for gaps this small (its cached Kelly
  even lands on the wrong side of zero).
- **Significance** — the bettors are scored on independent shoe streams (no common random numbers), so
  Kelly-vs-Flat uses an **unpaired** z-test (growth z ≈ 7, decisive; ruin p ≈ 0.04, marginal).
- **Identical terms** — same basic-strategy play, same shoe/reshuffle machinery, same session horizon for
  every bettor; the only thing that varies is the wager rule.
- **The oracle** (Chapter 3) is the positive control — the same network fed a *denoised* reward, to show
  the pipeline *can* learn Kelly when the signal is clean.